In [4]:
%pip -q install "numpy<2.0"
%pip -q install -U transformers
%pip -q install -U transformer_lens sae_lens
%pip -q install -U accelerate protobuf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 101.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.37.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires nu

In [1]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformer_lens import HookedTransformer
from sae_lens import SAE
from huggingface_hub import login
from google.colab import userdata

gc.collect()
torch.cuda.empty_cache()

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

MODEL_NAME = "google/gemma-2-2b"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"{MODEL_NAME} {device}")

hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map=device
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = HookedTransformer.from_pretrained(
    "gemma-2-2b",
    hf_model=hf_model,
    tokenizer=tokenizer,
    device=device,
    fold_ln=False,
    center_writing_weights=False,
    center_unembed=False,
    dtype=torch.float16
)


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded pretrained model gemma-2-2b into HookedTransformer


In [ ]:
class SteeringEngine:
    def __init__(self, model):
        self.model = model
        self.sae = None
        self.sae_layer = None

    def load_sae(self, release_name, sae_id, layer):
        print(f"using {sae_id}")
        self.sae = SAE.from_pretrained(
            release=release_name,
            sae_id=sae_id,
            device=device
        )
        self.sae_layer = layer
        print(f"layer {layer}"

    def _steering_hook(self, resid_pre, hook):
        steering_vector = self.sae.W_dec[self.target_feature_id]

        resid_pre[:, :] += steering_vector * self.steering_coefficient
        return resid_pre

    def generate(self, prompt, method="baseline", feature_id=None, strength=0.0):
        self.model.reset_hooks()

        if method == "baseline":
            return self.model.generate(prompt, max_new_tokens=20, verbose=False)

        if method == "prompted":
            augmented_prompt = f"You are a very positive assistant. {prompt}"
            return self.model.generate(augmented_prompt, max_new_tokens=20, verbose=False)

        if method == "steered":
            if self.sae is None:
                raise ValueError

            self.target_feature_id = feature_id
            self.steering_coefficient = strength

            hook_point = f"blocks.{self.sae_layer}.hook_resid_post"

            with self.model.hooks(fwd_hooks=[(hook_point, self._steering_hook)]):
                output = self.model.generate(prompt, max_new_tokens=20, verbose=False)
            return output

    def scan_features(self, prompt, feature_ids, strength=10.0):
      print(f"prompt: {prompt}")
      for fid in feature_ids:
        text = self.generate(prompt, method="steered", feature_id=fid, strength=strength)
        text = text.replace("\n", " ")
        print(f"Feature# {fid}: {text}")

engine = SteeringEngine(model)

In [4]:
prefix = "The movie was"

print(f"--- INPUT: '{prefix}' ---\n")

res_base = engine.generate(prefix, method="baseline")
print(f"[Baseline]: {res_base}")

res_prompt = engine.generate(prefix, method="prompted")
print(f"[Prompted]: {res_prompt}")

engine.load_sae(
    release_name="gemma-scope-2b-pt-res",
    sae_id="layer_12/width_16k/average_l0_82",
    layer=0
)

res_steer = engine.generate(prefix, method="steered", feature_id=1766, strength=10.0)
print(f"[Steered]:  {res_steer}")

--- INPUT: 'The movie was' ---

[Baseline]: The movie was amazing!

As I came into this movie I knew nothing about it, except for it was about
[Prompted]: You are a very positive assistant. The movie was pretty good and we laughed throughout. Well done! It was really a joy to work with you.
layer_12/width_16k/average_l0_82


layer_12/width_16k/average_l0_82/params.(…):   0%|          | 0.00/302M [00:00<?, ?B/s]

/tmp/ipython-input-3475517429.py:9: DeprecationWarning: Unpacking SAE objects is deprecated. SAE.from_pretrained() now returns only the SAE object. Use SAE.from_pretrained_with_cfg_and_sparsity() to get the config dict and sparsity as well.
  self.sae, _, _ = SAE.from_pretrained(


[Steered]:  The movie was shot in 7.1 and sound engineer Tom Johnson explains how they made the mixture sound full and
